## Reading data from Bronze layer

In [15]:
from pyspark.sql.functions import col, to_date, lower
df_bronze = spark.read.parquet("abfss://retail@pocdataforretail.dfs.core.windows.net/bronze/Jeena0317/ADLS-ADF-Spark-SQL-Synapse-Project/refs/heads/main/retail_transactions_bronze.parquet")


StatementMeta(pocsparkpool, 0, 16, Finished, Available, Finished, False)

In [17]:
df_bronze.count()

StatementMeta(pocsparkpool, 0, 18, Finished, Available, Finished, False)

120

In [18]:
display(df_bronze)

StatementMeta(pocsparkpool, 0, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 863059cd-98bf-431e-ac66-212ee60783cb)

## Silver layer

In [22]:


# Silver Layer - Filter purchase & clean data

df_silver = (
    df_bronze
    .filter(col("event_type") == "purchase")
    .dropna(subset=["customer_id", "amount"])
    .withColumn("event_date", to_date(col("event_timestamp")))
    .withColumn("payment_method", lower(col("payment_method")))
    .withColumn("amount", col("amount").cast("float"))
    .select(
        "event_id", "customer_id", "event_date", "product_id",
        "product_category", "payment_method", "amount", "location"
    )
)
print("Silver rows:", df_silver.count())
df_silver.printSchema()
df_silver.write.mode("overwrite").parquet("abfss://retail@pocdataforretail.dfs.core.windows.net/silver/")

StatementMeta(pocsparkpool, 0, 23, Finished, Available, Finished, False)

Silver rows: 41
root
 |-- event_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- amount: float (nullable = true)
 |-- location: string (nullable = true)



In [19]:
display(df_silver)

StatementMeta(pocsparkpool, 0, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c08995f9-3b41-48d2-ad93-9c966ba88731)

In [20]:
print(df_silver.count())
df_silver.show(5)

StatementMeta(pocsparkpool, 0, 21, Finished, Available, Finished, False)

41
+--------+-----------+----------+----------+----------------+--------------+------+---------+
|event_id|customer_id|event_date|product_id|product_category|payment_method|amount| location|
+--------+-----------+----------+----------+----------------+--------------+------+---------+
|   E1005|        C16|2024-05-27|     P1003|     Electronics|          cash|249.18|  Kolkata|
|   E1007|           |2024-05-28|     P1003|      Home Decor|   net banking|379.66|   Mumbai|
|   E1011|        C14|2024-05-24|     P1002|        Clothing|    debit card|162.41|  Kolkata|
|   E1016|         C6|2024-05-21|     P1007|      Home Decor|   net banking|164.06|Bangalore|
|   E1020|        C18|2024-05-06|     P1006|     Electronics|          cash|213.54|  Chennai|
+--------+-----------+----------+----------+----------------+--------------+------+---------+
only showing top 5 rows



## Gold layer transformation

In [25]:



# Gold Layer - Aggregates
from pyspark.sql.functions import sum, count, col

df_silver = spark.read.parquet("abfss://retail@pocdataforretail.dfs.core.windows.net/silver/")
display(df_silver)


StatementMeta(pocsparkpool, 0, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b0b1fda-c585-4727-a8c9-b36d5b16bb69)

In [27]:
df_silver.show()

df_daily_revenue = (
    df_silver.groupBy("event_date")
    .agg(sum("amount").alias("daily_revenue"), count("*").alias("total_purchases"))
)

df_daily_revenue.write.mode("overwrite").parquet("abfss://retail@pocdataforretail.dfs.core.windows.net/gold/")


StatementMeta(pocsparkpool, 0, 28, Finished, Available, Finished, False)

+--------+-----------+----------+----------+----------------+--------------+------+---------+
|event_id|customer_id|event_date|product_id|product_category|payment_method|amount| location|
+--------+-----------+----------+----------+----------------+--------------+------+---------+
|   E1005|        C16|2024-05-27|     P1003|     Electronics|          cash|249.18|  Kolkata|
|   E1007|           |2024-05-28|     P1003|      Home Decor|   net banking|379.66|   Mumbai|
|   E1011|        C14|2024-05-24|     P1002|        Clothing|    debit card|162.41|  Kolkata|
|   E1016|         C6|2024-05-21|     P1007|      Home Decor|   net banking|164.06|Bangalore|
|   E1020|        C18|2024-05-06|     P1006|     Electronics|          cash|213.54|  Chennai|
|   E1021|         C9|2024-05-23|     P1017|           Books|   net banking|311.96|  Kolkata|
|   E1023|         C3|2024-05-25|     P1007|      Home Decor|          cash|130.08|    Delhi|
|   E1026|         C7|2024-05-24|     P1010|       Groceries